# 🎯 IVE Member Segmentation - Fine-tune SAM 3

โปรเจค Text-to-Segment สำหรับ IVE Member ด้วยการ Fine-tune SAM 3

**Approach:**
1. Data Annotation ด้วย X-AnyLabeling → COCO Format
2. Fine-tune SAM 3 ด้วย LoRA
3. Inference บน Image/Video

**Labels:**
- `Wonyoung` - คลุมทั้งตัว
- `Wonyoung_shirt` - เฉพาะเสื้อ
- และ Member อื่นๆ

## 📦 Phase 0: Setup & Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers accelerate peft bitsandbytes
!pip install opencv-python matplotlib pillow tqdm
!pip install gradio
!pip install huggingface_hub

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import json
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import gradio as gr
from transformers import (
    SamModel, SamProcessor, SamImageProcessor,
    CLIPProcessor, CLIPModel
)
from peft import LoraConfig, get_peft_model, TaskType
from huggingface_hub import login

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 🔐 Hugging Face Login (สำหรับ SAM)

In [ ]:
# Login to Hugging Face (ถ้าจำเป็นสำหรับ SAM)
# login()  # Uncomment และใส่ token ถ้าต้องการ

## 📁 Phase 1: Data Preparation

**คำแนะนำก่อนใช้:**
1. ใช้ X-AnyLabeling ทำการตีกรอบและสร้าง mask สำหรับภาพ
2. Export เป็น COCO Format (annotations.json)
3. วางไฟล์ไว้ที่ `data/annotations.json` และ `data/images/`

In [ ]:
# สร้างโฟลเดอร์สำหรับเก็บข้อมูล
import os

os.makedirs('data/images', exist_ok=True)
os.makedirs('data/masks', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("โฟลเดอร์ที่สร้าง:")
print("- data/images/ : สำหรับเก็บรูปภาพที่ทำการ annotate")
print("- data/masks/ : สำหรับเก็บ mask ที่สร้างจาก annotation")
print("- models/ : สำหรับเก็บโมเดลที่ fine-tune")
print("- outputs/ : สำหรับเก็บผลลัพธ์")

In [ ]:
# Helper function สร้าง COCO format ตัวอย่าง (ถ้ายังไม่มี annotation)
# หรือใช้กับ Dataset ที่มีอยู่แล้ว

def create_sample_coco_annotation():
    """
    สร้างตัวอย่าง COCO annotation structure
    ใช้เป็นแม่แบบสำหรับการสร้าง annotations ของตัวเอง
    """
    coco_format = {
        "info": {
            "description": "IVE Member Segmentation Dataset",
            "version": "1.0",
            "year": 2024
        },
        "categories": [
            {"id": 1, "name": "Wonyoung", "supercategory": "person"},
            {"id": 2, "name": "Wonyoung_shirt", "supercategory": "clothing"},
            {"id": 3, "name": "Yujin", "supercategory": "person"},
            {"id": 4, "name": "Yujin_shirt", "supercategory": "clothing"},
            {"id": 5, "name": "Gaeul", "supercategory": "person"},
            {"id": 6, "name": "Rei", "supercategory": "person"},
            {"id": 7, "name": "Liz", "supercategory": "person"},
            {"id": 8, "name": "Leeseo", "supercategory": "person"}
        ],
        "images": [],
        "annotations": []
    }
    return coco_format

# แสดงโครงสร้าง categories ที่รองรับ
sample = create_sample_coco_annotation()
print("Categories ที่รองรับ:")
for cat in sample['categories']:
    print(f"  - {cat['id']}: {cat['name']} ({cat['supercategory']})")

### โหลดและแสดง Dataset ที่มีอยู่

In [ ]:
# ตรวจสอบ Dataset ที่มีอยู่
dataset_path = 'Dataset'

if os.path.exists(dataset_path):
    members = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    print(f"Found {len(members)} members:")
    for member in members:
        member_path = os.path.join(dataset_path, member)
        images = [f for f in os.listdir(member_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"  - {member}: {len(images)} images")
else:
    print(f"Dataset path '{dataset_path}' not found. กรุณาใส่รูปภาพสำหรับทำ annotation")

## 🔧 Phase 2: Data Preprocessing

แปลง COCO JSON เป็น PyTorch Dataset ที่ SAM 3 เข้าใจ

In [ ]:
class COCOSegmentationDataset(Dataset):
    """
    Dataset สำหรับโหลด COCO format segmentation data
    """
    def __init__(self, annotation_file, image_dir, processor, text_labels=None):
        """
        Args:
            annotation_file: path ไปยังไฟล์ annotations.json
            image_dir: path ไปยังโฟลเดอร์รูปภาพ
            processor: SamProcessor สำหรับ preprocess รูปภาพ
            text_labels: dict mapping category_id -> text prompt (e.g., {1: "Wonyoung"})
        """
        self.image_dir = Path(image_dir)
        self.processor = processor
        self.text_labels = text_labels or {}
        
        # โหลด COCO annotations
        if os.path.exists(annotation_file):
            with open(annotation_file, 'r') as f:
                self.coco_data = json.load(f)
        else:
            # สร้าง empty dataset ถ้ายังไม่มี annotations
            self.coco_data = create_sample_coco_annotation()
        
        # สร้าง mapping
        self.images = {img['id']: img for img in self.coco_data['images']}
        self.categories = {cat['id']: cat for cat in self.coco_data['categories']}
        
        # Group annotations by image
        self.image_annotations = {}
        for ann in self.coco_data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.image_annotations:
                self.image_annotations[img_id] = []
            self.image_annotations[img_id].append(ann)
        
        # รายการ image ids ที่มี annotation
        self.valid_image_ids = list(self.image_annotations.keys())
        
        print(f"Loaded {len(self.valid_image_ids)} images with annotations")
        print(f"Categories: {[cat['name'] for cat in self.coco_data['categories']]}")
    
    def polygon_to_mask(self, segmentation, height, width):
        """แปลง polygon segmentation เป็น binary mask"""
        mask = np.zeros((height, width), dtype=np.uint8)
        
        if isinstance(segmentation, list):
            # Polygon format
            for polygon in segmentation:
                pts = np.array(polygon).reshape(-1, 2).astype(np.int32)
                cv2.fillPoly(mask, [pts], 1)
        elif isinstance(segmentation, dict):
            # RLE format
            # TODO: Implement RLE decoding if needed
            pass
        
        return mask
    
    def __len__(self):
        return len(self.valid_image_ids)
    
    def __getitem__(self, idx):
        img_id = self.valid_image_ids[idx]
        img_info = self.images[img_id]
        
        # โหลดรูปภาพ
        img_path = self.image_dir / img_info['file_name']
        image = Image.open(img_path).convert('RGB')
        
        # สุ่มเลือก annotation หนึ่งตัวจากรูปภาพ
        annotations = self.image_annotations[img_id]
        ann = annotations[np.random.randint(len(annotations))]
        
        # สร้าง mask
        mask = self.polygon_to_mask(
            ann['segmentation'],
            img_info['height'],
            img_info['width']
        )
        
        # สร้าง text prompt
        cat_id = ann['category_id']
        if cat_id in self.text_labels:
            text = self.text_labels[cat_id]
        else:
            text = self.categories.get(cat_id, {}).get('name', 'person')
        
        # Preprocess ด้วย SAM processor
        inputs = self.processor(
            images=image,
            return_tensors="pt"
        )
        
        # Resize mask ให้ตรงกับขนาดที่ processor ใช้
        target_size = inputs['pixel_values'].shape[-2:]
        mask_resized = cv2.resize(mask, target_size[::-1], interpolation=cv2.INTER_NEAREST)
        
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'original_image': image,
            'mask': torch.tensor(mask_resized, dtype=torch.float32),
            'text': text,
            'category_id': cat_id
        }

print("COCOSegmentationDataset class defined ✓")

### สร้าง Dataset จากรูปภาพที่มีอยู่ (Auto-annotate ด้วย SAM)

In [ ]:
# ถ้ายังไม่มี annotations.json เราจะสร้าง dataset ง่ายๆ จากการวิเคราะห์รูปภาพก่อน
# หรือใช้ Grounding DINO + SAM เพื่อสร้าง initial annotations

def create_simple_dataset_from_images(image_dirs, output_annotation_file):
    """
    สร้าง COCO annotations จากรูปภาพในโฟลเดอร์
    โดยใช้ Grounding DINO + SAM สำหรับ auto-segmentation
    """
    coco_data = create_sample_coco_annotation()
    
    ann_id = 1
    img_id = 1
    
    # Mapping ชื่อโฟลเดอร์กับ category
    folder_to_category = {
        'Jang_Wonyoung': 1,
        'An_Yujin': 3,
        'Kim_Gaeul': 5,
        'Naoi_Rei': 6,
        'Kim_Jiwon': 7,
        'Lee_Hyunseo': 8
    }
    
    category_mapping = {
        1: 'Wonyoung',
        3: 'Yujin',
        5: 'Gaeul',
        6: 'Rei',
        7: 'Liz',
        8: 'Leeseo'
    }
    
    print("Processing images for dataset creation...")
    
    for folder_name, cat_id in folder_to_category.items():
        folder_path = os.path.join(image_dirs, folder_name)
        
        if not os.path.exists(folder_path):
            continue
        
        member_name = category_mapping.get(cat_id, folder_name)
        print(f"\nProcessing {member_name}...")
        
        image_files = [f for f in os.listdir(folder_path) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        
        for img_file in tqdm(image_files[:20], desc=member_name):  # จำกัด 20 รูปต่อคน
            try:
                img_path = os.path.join(folder_path, img_file)
                image = Image.open(img_path).convert('RGB')
                width, height = image.size
                
                # คัดลอกรูปไปยัง data/images/
                new_filename = f"{member_name}_{img_id:04d}.jpg"
                image.save(os.path.join('data/images', new_filename))
                
                # Add image info
                coco_data['images'].append({
                    'id': img_id,
                    'file_name': new_filename,
                    'width': width,
                    'height': height
                })
                
                # สร้าง dummy annotation (bbox ทั้งรูป - จะต้องมา annotate จริงทีหลัง)
                # ในการใช้งานจริง ควรใช้ X-AnyLabeling เพื่อสร้าง mask ที่ถูกต้อง
                
                img_id += 1
                
            except Exception as e:
                print(f"Error processing {img_file}: {e}")
    
    # Save annotations
    with open(output_annotation_file, 'w') as f:
        json.dump(coco_data, f, indent=2)
    
    print(f"\n✓ Created dataset with {len(coco_data['images'])} images")
    print(f"✓ Saved to {output_annotation_file}")
    
    return coco_data

# สร้าง dataset (run ครั้งแรกเท่านั้น)
annotation_file = 'data/annotations.json'

if not os.path.exists(annotation_file) or os.path.getsize(annotation_file) < 100:
    coco_data = create_simple_dataset_from_images('Dataset', annotation_file)
else:
    print(f"Annotation file already exists: {annotation_file}")
    with open(annotation_file, 'r') as f:
        coco_data = json.load(f)
    print(f"Loaded {len(coco_data['images'])} images")

## 🤖 Phase 3: Load SAM 3 Model & Setup LoRA

In [ ]:
# Model configuration
MODEL_NAME = "facebook/sam-vit-huge"  # หรือ "facebook/sam-vit-large", "facebook/sam-vit-base"
# หมายเหตุ: SAM 3 อาจยังไม่พร้อมใช้งานบน HuggingFace อย่างเป็นทางการ
# ถ้า SAM 3 ยังไม่มี ให้ใช้ SAM 1 หรือ SAM 2 แทนก่อน

# ลองโหลดโมเดล
try:
    from transformers import SamModel, SamProcessor
    
    print(f"Loading {MODEL_NAME}...")
    processor = SamProcessor.from_pretrained(MODEL_NAME)
    model = SamModel.from_pretrained(MODEL_NAME)
    print("✓ Model loaded successfully")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTrying alternative approach...")
    
    # Fallback to segment-anything ถ้า transformers ไม่ work
    try:
        !pip install segment-anything
        from segment_anything import sam_model_registry, SamPredictor
        
        # Download checkpoint ถ้ายังไม่มี
        sam_checkpoint = "sam_vit_h.pth"
        if not os.path.exists(sam_checkpoint):
            !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h.pth
        
        sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint)
        sam.to(device="cuda" if torch.cuda.is_available() else "cpu")
        
        print("✓ SAM loaded via segment-anything")
        model = sam
        processor = None
        
    except Exception as e2:
        print(f"Alternative also failed: {e2}")
        print("Please install segment-anything or check model availability")

In [ ]:
# Setup LoRA for fine-tuning

def setup_lora_model(model, r=8, lora_alpha=16, target_modules=None):
    """
    Setup LoRA adapters สำหรับ SAM model
    โดยจะ fine-tune เฉพาะส่วน prompt encoder และ mask decoder
    """
    
    # Freeze all base model parameters
    for param in model.parameters():
        param.requires_grad = False
    
    # กำหนด target modules สำหรับ LoRA
    # สำหรับ SAM เราจะ target ที่ prompt encoder และ mask decoder
    if target_modules is None:
        # กำหนด modules ที่จะใส่ LoRA
        target_modules = [
            "prompt_encoder",
            "mask_decoder",
        ]
    
    # LoRA configuration
    lora_config = LoraConfig(
        r=r,  # rank
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )
    
    # ใช้ PEFT ใส่ LoRA เข้าไปใน model
    # Note: สำหรับ SAM ที่ไม่ใช่ standard transformer เราอาจต้องทำ manually
    
    # วิธีที่ 2: Manual LoRA implementation สำหรับ SAM
    class LoRALayer(nn.Module):
        def __init__(self, in_features, out_features, rank=r, alpha=lora_alpha):
            super().__init__()
            self.lora_A = nn.Parameter(torch.randn(in_features, rank) * 0.01)
            self.lora_B = nn.Parameter(torch.zeros(rank, out_features))
            self.alpha = alpha
            self.scaling = alpha / rank
        
        def forward(self, x):
            return x @ self.lora_A @ self.lora_B * self.scaling
    
    print("LoRA configuration:")
    print(f"  Rank (r): {r}")
    print(f"  Alpha: {lora_alpha}")
    print(f"  Scaling: {lora_alpha/r}")
    
    return model, LoRALayer

# ถ้าใช้ transformers SAM
if processor is not None:
    model, LoRALayer = setup_lora_model(model)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"\n✓ Model moved to {device}")

### Loss Functions สำหรับ Segmentation

In [ ]:
class SegmentationLoss(nn.Module):
    """
    Combined Loss สำหรับ Segmentation
    - Dice Loss: จัดการ class imbalance
    - Focal Loss: Focus บน hard examples
    """
    
    def __init__(self, dice_weight=0.5, focal_weight=0.5, focal_gamma=2.0):
        super().__init__()
        self.dice_weight = dice_weight
        self.focal_weight = focal_weight
        self.focal_gamma = focal_gamma
        self.bce = nn.BCEWithLogitsLoss(reduction='none')
    
    def dice_loss(self, pred, target, smooth=1e-5):
        """Dice Loss for segmentation"""
        pred = torch.sigmoid(pred)
        
        # Flatten
        pred = pred.view(-1)
        target = target.view(-1)
        
        intersection = (pred * target).sum()
        union = pred.sum() + target.sum()
        
        dice = (2. * intersection + smooth) / (union + smooth)
        return 1 - dice
    
    def focal_loss(self, pred, target):
        """Focal Loss สำหรับจัดการ hard examples"""
        bce_loss = self.bce(pred, target)
        pred_prob = torch.sigmoid(pred)
        
        # Focal weight
        p_t = pred_prob * target + (1 - pred_prob) * (1 - target)
        focal_weight = (1 - p_t) ** self.focal_gamma
        
        return (focal_weight * bce_loss).mean()
    
    def forward(self, pred, target):
        dice = self.dice_loss(pred, target)
        focal = self.focal_loss(pred, target)
        
        total_loss = self.dice_weight * dice + self.focal_weight * focal
        
        return {
            'total': total_loss,
            'dice': dice,
            'focal': focal
        }

# Test loss function
criterion = SegmentationLoss()
print("✓ SegmentationLoss defined")

## 🏋️ Phase 3.5: Training Loop (Fine-tuning)

In [ ]:
def train_sam_model(
    model,
    processor,
    train_dataset,
    num_epochs=10,
    batch_size=2,
    learning_rate=1e-4,
    device='cuda'
):
    """
    Training loop สำหรับ fine-tune SAM
    """
    
    model.train()
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=learning_rate,
        weight_decay=1e-4
    )
    
    criterion = SegmentationLoss()
    
    # DataLoader
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )
    
    history = {
        'train_loss': [],
        'dice_loss': [],
        'focal_loss': []
    }
    
    for epoch in range(num_epochs):
        epoch_losses = []
        epoch_dice = []
        epoch_focal = []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        
        for batch_idx, batch in enumerate(pbar):
            try:
                # Get data
                pixel_values = batch['pixel_values'].to(device)
                masks = batch['mask'].to(device)
                texts = batch['text']
                
                # Forward pass ผ่าน SAM
                # Note: จริงๆ แล้ว SAM ต้องการ input points หรือ boxes ด้วย
                # แต่สำหรับ text-to-segment เราจะต้องมีวิธีแปลง text เป็น prompt
                
                # สำหรับการเทรนจริง ควรใช้: 
                # 1. Grounding DINO หรือ Owl-ViT สร้าง box จาก text
                # 2. ใช้ box เป็น prompt ให้ SAM
                
                # ตัวอย่าง simplified forward (จะต้องปรับตาม architecture จริง)
                outputs = model(
                    pixel_values=pixel_values,
                    multimask_output=False
                )
                
                # คำนวณ loss (ปรับตาม output format จริงของ SAM)
                pred_masks = outputs.pred_masks if hasattr(outputs, 'pred_masks') else outputs[0]
                
                # Resize prediction ให้ตรงกับ ground truth
                pred_masks = F.interpolate(
                    pred_masks,
                    size=masks.shape[-2:],
                    mode='bilinear',
                    align_corners=False
                )
                
                # Calculate loss
                loss_dict = criterion(pred_masks, masks)
                loss = loss_dict['total']
                
                # Backward
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                # Record
                epoch_losses.append(loss.item())
                epoch_dice.append(loss_dict['dice'].item())
                epoch_focal.append(loss_dict['focal'].item())
                
                # Update progress bar
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'dice': f"{loss_dict['dice'].item():.4f}"
                })
                
            except Exception as e:
                print(f"\nError in batch {batch_idx}: {e}")
                continue
        
        # Epoch summary
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0
        avg_dice = np.mean(epoch_dice) if epoch_dice else 0
        avg_focal = np.mean(epoch_focal) if epoch_focal else 0
        
        history['train_loss'].append(avg_loss)
        history['dice_loss'].append(avg_dice)
        history['focal_loss'].append(avg_focal)
        
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"  Loss: {avg_loss:.4f}")
        print(f"  Dice Loss: {avg_dice:.4f}")
        print(f"  Focal Loss: {avg_focal:.4f}")
    
    return history

print("✓ Training function defined")

### Run Training (ถ้ามี annotated data)

In [ ]:
# ตรวจสอบว่ามี annotations ที่สมบูรณ์หรือไม่
# (ต้องมี mask annotations จริงๆ ไม่ใช่แค่รายการรูปภาพ)

has_annotations = len(coco_data.get('annotations', [])) > 0

if has_annotations and processor is not None:
    print("Starting training with annotated data...")
    
    # Text labels mapping
    text_labels = {
        1: "Wonyoung",
        2: "Wonyoung's shirt",
        3: "Yujin",
        4: "Yujin's shirt",
        5: "Gaeul",
        6: "Rei",
        7: "Liz",
        8: "Leeseo"
    }
    
    # Create dataset
    train_dataset = COCOSegmentationDataset(
        annotation_file='data/annotations.json',
        image_dir='data/images',
        processor=processor,
        text_labels=text_labels
    )
    
    if len(train_dataset) > 0:
        # Train
        history = train_sam_model(
            model=model,
            processor=processor,
            train_dataset=train_dataset,
            num_epochs=5,
            batch_size=2,
            learning_rate=1e-4,
            device=device
        )
        
        # Save model
        torch.save(model.state_dict(), 'models/sam_finetuned.pth')
        print("\n✓ Model saved to models/sam_finetuned.pth")
    else:
        print("Dataset is empty. Please add annotations first.")
else:
    print("No annotations found or model not loaded.")
    print("\nTo train the model:")
    print("1. Use X-AnyLabeling to annotate images")
    print("2. Export to COCO format")
    print("3. Run training cell again")

## 🔍 Phase 4: Inference Functions

In [ ]:
class IVESegmentationInference:
    """
    Inference class สำหรับ IVE Member Segmentation
    รองรับทั้ง Identity และ Possession prompts
    """
    
    def __init__(self, model, processor, device='cuda'):
        self.model = model
        self.processor = processor
        self.device = device
        self.model.eval()
        
        # Member mapping
        self.members = {
            'wonyoung': ['wonyoung', 'jang wonyoung', '장원영'],
            'yujin': ['yujin', 'an yujin', '안유진'],
            'gaeul': ['gaeul', 'kim gaeul', '가을'],
            'rei': ['rei', 'naoi rei', '레이'],
            'liz': ['liz', 'kim jiwon', '김지원', '리즈'],
            'leeseo': ['leeseo', 'lee hyunseo', '이현서', '이서']
        }
    
    def parse_prompt(self, prompt):
        """
        Parse prompt เพื่อแยกระหว่าง Identity และ Possession
        """
        prompt_lower = prompt.lower().strip()
        
        # Check for possession (e.g., "Wonyoung's shirt")
        possession_keywords = ["'s", "ของ", "wear", "wearing"]
        is_possession = any(kw in prompt_lower for kw in possession_keywords)
        
        # Extract member name
        target_member = None
        object_name = None
        
        for member, aliases in self.members.items():
            for alias in aliases:
                if alias in prompt_lower:
                    target_member = member
                    break
            if target_member:
                break
        
        # Extract object (for possession)
        if is_possession and target_member:
            # Remove member name and possession keywords
            object_name = prompt_lower
            for alias in self.members.get(target_member, []):
                object_name = object_name.replace(alias, '')
            for kw in possession_keywords:
                object_name = object_name.replace(kw, '')
            object_name = object_name.strip()
        
        return {
            'is_possession': is_possession,
            'member': target_member,
            'object': object_name,
            'original': prompt
        }
    
    def segment_image(self, image, prompt, box=None):
        """
        Segment image based on prompt
        """
        parsed = self.parse_prompt(prompt)
        
        # Convert image to PIL if needed
        if isinstance(image, str):
            image = Image.open(image).convert('RGB')
        elif isinstance(image, np.ndarray):
            image = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        
        # Prepare inputs
        if box is not None:
            # Use provided box
            inputs = self.processor(
                images=image,
                input_boxes=[[box]],
                return_tensors="pt"
            )
        else:
            # Use image only (will need point prompts or boxes)
            inputs = self.processor(
                images=image,
                return_tensors="pt"
            )
        
        # Move to device
        inputs = {k: v.to(self.device) if isinstance(v, torch.Tensor) else v 
                 for k, v in inputs.items()}
        
        # Inference
        with torch.no_grad():
            outputs = self.model(**inputs, multimask_output=False)
        
        # Get mask
        mask = outputs.pred_masks.squeeze().cpu().numpy()
        mask = (mask > 0).astype(np.uint8) * 255
        
        return {
            'mask': mask,
            'parsed_prompt': parsed,
            'image': image
        }
    
    def visualize_result(self, image, mask, alpha=0.5):
        """
        Overlay mask บนรูปภาพ
        """
        if isinstance(image, Image.Image):
            image = np.array(image)
        
        # Create colored mask
        colored_mask = np.zeros_like(image)
        colored_mask[mask > 0] = [255, 0, 0]  # สีแดง
        
        # Overlay
        overlay = cv2.addWeighted(image, 1, colored_mask, alpha, 0)
        
        return overlay

# Initialize inference
if processor is not None:
    inference = IVESegmentationInference(model, processor, device)
    print("✓ Inference class initialized")

### Video Processing

In [ ]:
def process_video(
    inference,
    video_path,
    prompt,
    output_path=None,
    box=None,
    sample_rate=1
):
    """
    Process video frame by frame
    
    Args:
        inference: IVESegmentationInference instance
        video_path: path to input video
        prompt: text prompt (e.g., "Wonyoung")
        output_path: path for output video (optional)
        box: bounding box [x1, y1, x2, y2] (optional)
        sample_rate: process every N frames
    """
    
    if output_path is None:
        output_path = f"outputs/output_{Path(video_path).stem}.mp4"
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Cannot open video {video_path}")
        return None
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"Video info: {width}x{height} @ {fps}fps, {total_frames} frames")
    
    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps // sample_rate, (width, height))
    
    frame_count = 0
    processed_count = 0
    
    with tqdm(total=total_frames // sample_rate, desc="Processing video") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            frame_count += 1
            
            # Skip frames ตาม sample_rate
            if frame_count % sample_rate != 0:
                continue
            
            try:
                # Segment frame
                result = inference.segment_image(frame, prompt, box)
                mask = result['mask']
                
                # Visualize
                overlay = inference.visualize_result(frame, mask)
                overlay_bgr = cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR)
                
                # Write
                out.write(overlay_bgr)
                processed_count += 1
                
                pbar.update(1)
                
            except Exception as e:
                print(f"\nError processing frame {frame_count}: {e}")
                # Write original frame if error
                out.write(frame)
    
    cap.release()
    out.release()
    
    print(f"\n✓ Processed {processed_count} frames")
    print(f"✓ Output saved to: {output_path}")
    
    return output_path

print("✓ Video processing function defined")

## 🎨 Phase 5: Gradio UI

In [ ]:
def create_gradio_interface(inference):
    """
    สร้าง Gradio interface สำหรับ demo
    """
    
    def process_image(image, prompt):
        """Process image and return result"""
        try:
            result = inference.segment_image(image, prompt)
            mask = result['mask']
            overlay = inference.visualize_result(result['image'], mask)
            
            return overlay, mask
        except Exception as e:
            return None, f"Error: {str(e)}"
    
    def process_video_gradio(video, prompt, sample_rate):
        """Process video and return result"""
        try:
            output_path = process_video(
                inference,
                video,
                prompt,
                sample_rate=int(sample_rate)
            )
            return output_path
        except Exception as e:
            return f"Error: {str(e)}"
    
    # CSS สำหรับ styling
    css = """
    .gradio-container {
        max-width: 1200px;
        margin: 0 auto;
    }
    .header {
        text-align: center;
        margin-bottom: 20px;
    }
    .prompt-examples {
        background: #f5f5f5;
        padding: 15px;
        border-radius: 8px;
        margin: 10px 0;
    }
    """
    
    with gr.Blocks(css=css, title="IVE Segmentation") as demo:
        
        # Header
        gr.Markdown("""
        # 🎯 IVE Member Segmentation
        ## Text-to-Segment with Fine-tuned SAM
        
        ระบบ segmentation สำหรับสมาชิกวง IVE รองรับทั้ง Identity และ Possession prompts
        """)
        
        with gr.Tab("📷 Image Segmentation"):
            with gr.Row():
                with gr.Column():
                    input_image = gr.Image(label="Upload Image", type="pil")
                    prompt_input = gr.Textbox(
                        label="Prompt",
                        placeholder="e.g., Wonyoung, Yujin's shirt",
                        value="Wonyoung"
                    )
                    
                    with gr.Accordion("💡 Prompt Examples", open=False):
                        gr.Markdown("""
                        **Identity:**
                        - `Wonyoung` - หาคนชื่อวอนยอง
                        - `Yujin` - หาคนชื่อยูจิน
                        - `Gaeul`, `Rei`, `Liz`, `Leeseo`
                        
                        **Possession:**
                        - `Wonyoung's shirt` - เสื้อของวอนยอง
                        - `Yujin's dress` - ชุดของยูจิน
                        """)
                    
                    segment_btn = gr.Button("🚀 Segment", variant="primary")
                
                with gr.Column():
                    output_image = gr.Image(label="Result")
                    output_mask = gr.Image(label="Mask")
            
            segment_btn.click(
                fn=process_image,
                inputs=[input_image, prompt_input],
                outputs=[output_image, output_mask]
            )
        
        with gr.Tab("🎬 Video Segmentation"):
            with gr.Row():
                with gr.Column():
                    input_video = gr.Video(label="Upload Video")
                    video_prompt = gr.Textbox(
                        label="Prompt",
                        placeholder="e.g., Wonyoung",
                        value="Wonyoung"
                    )
                    sample_rate = gr.Slider(
                        minimum=1,
                        maximum=10,
                        value=1,
                        step=1,
                        label="Sample Rate (process every N frames)"
                    )
                    process_video_btn = gr.Button("🎥 Process Video", variant="primary")
                
                with gr.Column():
                    output_video = gr.Video(label="Output Video")
            
            process_video_btn.click(
                fn=process_video_gradio,
                inputs=[input_video, video_prompt, sample_rate],
                outputs=output_video
            )
        
        gr.Markdown("""
        ---
        **Note:** สำหรับการใช้งานจริง แนะนำให้ fine-tune โมเดลกับ annotated data ก่อน
        """)
    
    return demo

# Create and launch interface
if processor is not None:
    demo = create_gradio_interface(inference)
    print("✓ Gradio interface created")
    print("\nLaunch the interface by running: demo.launch()")
else:
    print("Model not loaded. Cannot create interface.")

In [ ]:
# Launch Gradio
if processor is not None:
    demo.launch(share=True, debug=True)

## 🧪 Testing & Demo

In [ ]:
# Test with sample image (ถ้ามี)
def test_segmentation(image_path, prompt):
    """
    Test segmentation on a single image
    """
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return
    
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Segment
    result = inference.segment_image(image, prompt)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(result['mask'], cmap='gray')
    axes[1].set_title(f'Mask: {prompt}')
    axes[1].axis('off')
    
    overlay = inference.visualize_result(image, result['mask'])
    axes[2].imshow(overlay)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return result

# ทดสอบกับรูปจาก Dataset
sample_images = []
for member in ['Jang_Wonyoung', 'An_Yujin']:
    member_path = f'Dataset/{member}'
    if os.path.exists(member_path):
        images = [f for f in os.listdir(member_path) if f.endswith(('.jpg', '.png'))]
        if images:
            sample_images.append(os.path.join(member_path, images[0]))

if sample_images and processor is not None:
    print("Testing with sample images...")
    test_segmentation(sample_images[0], "person")

## 💾 Save/Load Fine-tuned Model

In [ ]:
# Save fine-tuned model
def save_model(model, processor, save_path='models/sam_ive_finetuned'):
    """
    Save fine-tuned model
    """
    os.makedirs(save_path, exist_ok=True)
    
    # Save model
    model.save_pretrained(save_path)
    processor.save_pretrained(save_path)
    
    print(f"Model saved to {save_path}")

# Load fine-tuned model
def load_model(load_path='models/sam_ive_finetuned'):
    """
    Load fine-tuned model
    """
    processor = SamProcessor.from_pretrained(load_path)
    model = SamModel.from_pretrained(load_path)
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    
    return model, processor, device

print("Save/Load functions defined")

## 📊 Training Visualization (ถ้ามีการเทรน)

In [ ]:
# Plot training history
def plot_training_history(history):
    """
    Plot training loss curves
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Total Loss
    axes[0].plot(epochs, history['train_loss'], 'b-', linewidth=2)
    axes[0].set_title('Total Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    
    # Dice Loss
    axes[1].plot(epochs, history['dice_loss'], 'r-', linewidth=2)
    axes[1].set_title('Dice Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True)
    
    # Focal Loss
    axes[2].plot(epochs, history['focal_loss'], 'g-', linewidth=2)
    axes[2].set_title('Focal Loss')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Loss')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

# ถ้ามี history จากการเทรน
if 'history' in locals():
    plot_training_history(history)

---

## 📝 Summary

Notebook นี้ประกอบด้วย:

1. **Phase 0**: Setup & Dependencies
2. **Phase 1**: Data Preparation (COCO Format)
3. **Phase 2**: Data Preprocessing
4. **Phase 3**: Model Fine-tuning with LoRA
5. **Phase 4**: Inference Functions
6. **Phase 5**: Gradio UI

**ขั้นตอนถัดไป:**
1. ใช้ X-AnyLabeling สร้าง annotations ที่สมบูรณ์
2. Run training loop
3. Test กับรูปภาพและวิดีโอ
4. Deploy Gradio UI